# Ноутбук 2 сборка WB датасета

In [1]:
# import sys; !{sys.executable} -m pip install -q anthropic datasets scikit-learn pandas matplotlib

In [2]:
import json
import os
import random
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE_DIR = Path('/Users/yana/Курсовая 3 курс')
os.chdir(BASE_DIR)
print('Рабочая папка:', BASE_DIR)


Рабочая папка: /Users/yana/Курсовая 3 курс


## Загрузка реальных WB-отзывов

In [3]:
def normalize_text(text: str) -> str:
    text = str(text or '').replace('\xa0', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def load_real_wb_reviews(limit=700, min_chars=40, max_chars=450):
    ds_wb = load_dataset('nyuuzyou/wb-feedbacks', split='train', streaming=True)
    rows, seen = [], set()

    for item in ds_wb:
        text = normalize_text(item.get('text'))
        if not (min_chars <= len(text) <= max_chars):
            continue
        if text.lower() in seen:
            continue
        seen.add(text.lower())

        rows.append({
            'real_id': f'wb_real_{len(rows):05d}',
            'text': text,
            'rating': item.get('productValuation'),
            'product_name': item.get('productName') or item.get('name') or '',
            'category': item.get('subjectName') or item.get('category') or '',
        })
        if len(rows) >= limit:
            break

    return pd.DataFrame(rows)

real_wb_df = load_real_wb_reviews(limit=700)
print('Загружено реальных отзывов:', len(real_wb_df))
display(real_wb_df.head(3))


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Загружено реальных отзывов: 700


,real_id,text,rating,product_name,category
0,wb_real_00000,"5 попыток было испечь хлеб с этой муки, и один...",1,,
1,wb_real_00001,1 звезда за то что нет даты изготовления и пит...,1,,
2,wb_real_00002,"2 недели уже стоит, так ничего и не взошло",1,,


## Контроль поверхностных сигналов

In [4]:
POSITIVE_MARKERS = [
    'отлично', 'отличный', 'отличная', 'супер', 'класс', 'понравилось',
    'понравился', 'доволен', 'довольна', 'хорошо', 'нравится', 'удобно'
]

NEGATIVE_MARKERS = [
    ' но ', 'однако', 'зато', 'хотя', 'жаль', 'к сожалению', 'минус',
    'недостаток', 'плохо', 'не понрав', 'разочар', 'брак', 'дефект',
    'сломал', 'порвал', 'не подош', 'маловат', 'великоват', 'возврат'
]

TEMPLATE_PHRASES = [
    'всем рекомендую', 'однозначно рекомендую', 'не пожалеете',
    'качество на высоте', 'продавцу спасибо', 'буду брать еще',
    'буду заказывать еще', 'в подарок', 'идеальный товар',
    'выше всяких похвал', 'на все сто', 'покупкой довольна',
    'покупкой доволен'
]


def count_markers(text, markers):
    t = f' {normalize_text(text).lower()} '
    return sum(1 for marker in markers if marker in t)


def sentiment_bucket(text):
    pos = count_markers(text, POSITIVE_MARKERS)
    neg = count_markers(text, NEGATIVE_MARKERS)
    if pos and neg:
        return 'mixed'
    if neg:
        return 'negative'
    if pos:
        return 'positive'
    return 'neutral'


def pair_audit(real_text, fake_text):
    real_text = normalize_text(real_text)
    fake_text = normalize_text(fake_text)
    ratio = len(fake_text) / max(len(real_text), 1)
    return {
        'real_len': len(real_text),
        'fake_len': len(fake_text),
        'len_ratio': ratio,
        'real_bucket': sentiment_bucket(real_text),
        'fake_bucket': sentiment_bucket(fake_text),
        'real_template_count': count_markers(real_text, TEMPLATE_PHRASES),
        'fake_template_count': count_markers(fake_text, TEMPLATE_PHRASES),
        'real_negative_count': count_markers(real_text, NEGATIVE_MARKERS),
        'fake_negative_count': count_markers(fake_text, NEGATIVE_MARKERS),
        'real_positive_count': count_markers(real_text, POSITIVE_MARKERS),
        'fake_positive_count': count_markers(fake_text, POSITIVE_MARKERS),
    }


def is_quality_pair(real_text, fake_text):
    audit = pair_audit(real_text, fake_text)
    length_ok = 0.75 <= audit['len_ratio'] <= 1.30
    template_ok = audit['fake_template_count'] <= 1
    sentiment_ok = not (
        audit['real_bucket'] in {'negative', 'mixed'}
        and audit['fake_bucket'] == 'positive'
        and audit['fake_negative_count'] == 0
    )
    return length_ok and template_ok and sentiment_ok


## Генерация парных fake-отзывов

In [5]:
PROMPT_PAIRED = """
Ты создаёшь данные для исследования методов NLP по выявлению fake/заказных отзывов в e-commerce.

Ниже дан реальный отзыв покупателя Wildberries:
"{real_review}"

Напиши искусственно подготовленный заказной вариант отзыва про тот же товар.
Важно: fake-отзыв НЕ должен быть очевидной рекламой.

Правила:
1. Сохрани категорию товара и основные детали использования.
2. Сохрани общий эмоциональный тон: если исходник негативный или смешанный, fake тоже может содержать мелкое замечание, сомнение или нейтральную деталь.
3. Длина должна быть близкой к исходнику: примерно 75-130% от длины исходного текста.
4. Не добавляй подарок, бонус, благодарность продавцу, если этого не было в исходнике.
5. Не используй шаблонные фразы: "всем рекомендую", "не пожалеете", "качество на высоте", "просто супер", "буду брать ещё".
6. Пиши живым русским языком WB-отзыва: допускаются разговорность, небольшие пунктуационные ошибки, короткие фразы.
7. Не делай текст слишком грамотным, гладким или маркетинговым.
8. Не копируй исходник дословно: измени формулировки, но сохрани тему.

Верни только текст fake-отзыва. Без пояснений, кавычек и JSON.
""".strip()

GENERATOR_MODEL = os.getenv('ANTHROPIC_MODEL', 'claude-sonnet-4-5')
PAIRED_PATH = BASE_DIR / 'paired_reviews.json'


def get_anthropic_client():
    api_key = os.getenv('ANTHROPIC_API_KEY')
    if not api_key:
        from getpass import getpass
        api_key = getpass('Вставь Anthropic API key. Ввод скрыт и не сохранится в ноутбуке: ').strip()
        if not api_key:
            raise RuntimeError('ANTHROPIC_API_KEY не задан. Генерация fake-отзывов невозможна без ключа.')
        os.environ['ANTHROPIC_API_KEY'] = api_key
    import anthropic
    return anthropic.Anthropic(api_key=api_key)


def generate_fake_review(client, real_text, max_tokens=450):
    resp = client.messages.create(
        model=GENERATOR_MODEL,
        max_tokens=max_tokens,
        temperature=0.9,
        messages=[{
            'role': 'user',
            'content': PROMPT_PAIRED.format(real_review=real_text)
        }]
    )
    return normalize_text(resp.content[0].text)


In [6]:
GENERATE_NEW_PAIRS = False
TARGET_PAIRS = 500
MAX_ATTEMPTS_PER_REVIEW = 3
SLEEP_SECONDS = 0.25

if GENERATE_NEW_PAIRS:
    client = get_anthropic_client()
    paired_data = []
    failed = 0

    sample_reals = real_wb_df.sample(
        n=min(TARGET_PAIRS, len(real_wb_df)),
        random_state=SEED
    ).reset_index(drop=True)

    for i, row in sample_reals.iterrows():
        real_text = row['text']
        accepted_fake = None

        for attempt in range(1, MAX_ATTEMPTS_PER_REVIEW + 1):
            try:
                fake_text = generate_fake_review(client, real_text)
                if fake_text != real_text and is_quality_pair(real_text, fake_text):
                    accepted_fake = fake_text
                    break
            except Exception as e:
                failed += 1
                if failed <= 5:
                    print('Ошибка генерации:', repr(e))
                break
            time.sleep(SLEEP_SECONDS)

        if accepted_fake:
            paired_data.append({
                'pair_id': f'wb_pair_{len(paired_data):05d}',
                'real': real_text,
                'fake': accepted_fake,
                'real_id': row.get('real_id', ''),
                'rating': row.get('rating'),
                'product_name': row.get('product_name', ''),
                'category': row.get('category', ''),
                'generator_model': GENERATOR_MODEL,
                'prompt_version': 'paired_v2_tone_matched',
                **pair_audit(real_text, accepted_fake),
            })

        if (i + 1) % 25 == 0:
            print(f'Обработано: {i+1}/{len(sample_reals)} | принято пар: {len(paired_data)} | ошибок: {failed}')

    with open(PAIRED_PATH, 'w', encoding='utf-8') as f:
        json.dump(paired_data, f, ensure_ascii=False, indent=2)

    print('Сохранено:', PAIRED_PATH.name)
    print('Принято качественных пар:', len(paired_data))
else:
    with open(PAIRED_PATH, encoding='utf-8') as f:
        paired_data = json.load(f)
    print('Загружено пар:', len(paired_data))


Вставь Anthropic API key. Ввод скрыт и не сохранится в ноутбуке:  ········


Обработано: 25/500 | принято пар: 1 | ошибок: 0
Обработано: 50/500 | принято пар: 6 | ошибок: 0
Обработано: 75/500 | принято пар: 11 | ошибок: 0
Обработано: 100/500 | принято пар: 14 | ошибок: 0
Обработано: 125/500 | принято пар: 17 | ошибок: 0
Обработано: 150/500 | принято пар: 21 | ошибок: 0
Обработано: 175/500 | принято пар: 24 | ошибок: 0
Обработано: 200/500 | принято пар: 25 | ошибок: 0
Обработано: 225/500 | принято пар: 27 | ошибок: 0
Обработано: 250/500 | принято пар: 29 | ошибок: 0
Обработано: 275/500 | принято пар: 32 | ошибок: 0
Обработано: 300/500 | принято пар: 32 | ошибок: 0
Обработано: 325/500 | принято пар: 34 | ошибок: 0
Обработано: 350/500 | принято пар: 38 | ошибок: 0
Обработано: 375/500 | принято пар: 43 | ошибок: 0
Обработано: 400/500 | принято пар: 46 | ошибок: 0
Обработано: 425/500 | принято пар: 47 | ошибок: 0
Обработано: 450/500 | принято пар: 50 | ошибок: 0
Обработано: 475/500 | принято пар: 56 | ошибок: 0
Обработано: 500/500 | принято пар: 56 | ошибок: 0
Сохра

In [11]:
import json, time
from pathlib import Path

RESUME_TARGET_PAIRS = 150
MAX_REVIEWS_TO_TRY = 500
SLEEP_SECONDS = 0.1

PAIRED_PATH = Path("paired_reviews.json")

with open(PAIRED_PATH, encoding="utf-8") as f:
    paired_data = json.load(f)

print("Уже есть пар:", len(paired_data))

used_texts = {
    normalize_text(p.get("real", "")).lower()
    for p in paired_data
}

candidate_reals = real_wb_df[
    ~real_wb_df["text"].map(lambda x: normalize_text(x).lower()).isin(used_texts)
].copy()

print("Кандидатов до sample:", len(candidate_reals))

candidate_reals = candidate_reals.sample(
    n=min(MAX_REVIEWS_TO_TRY, len(candidate_reals)),
    random_state=SEED + len(paired_data)
).reset_index(drop=True)

print("Кандидатов в этом запуске:", len(candidate_reals))
print("Цель:", RESUME_TARGET_PAIRS)

client = get_anthropic_client()
start_count = len(paired_data)

for i, row in candidate_reals.iterrows():
    if len(paired_data) >= RESUME_TARGET_PAIRS:
        break

    real_text = row["text"]

    try:
        # Без категорий: просто генерируем по тексту реального отзыва
        fake_text = generate_fake_review(client, real_text)
        fake_text = normalize_text(fake_text)

        audit = pair_audit(real_text, fake_text)

        length_ok = 0.50 <= audit["len_ratio"] <= 2.00
        template_ok = audit["fake_template_count"] <= 3
        not_duplicate = fake_text != real_text
        not_too_short = len(fake_text) >= 30

        if not_duplicate and not_too_short and length_ok and template_ok:
            paired_data.append({
                "pair_id": f"wb_pair_{len(paired_data):05d}",
                "real": real_text,
                "fake": fake_text,
                "real_id": row.get("real_id", ""),
                "rating": row.get("rating"),
                "product_name": "",
                "category": "",
                "generator_model": GENERATOR_MODEL,
                "prompt_version": "resume_no_categories_relaxed",
                **audit,
            })

    except Exception as e:
        print("Ошибка:", repr(e))

    if (i + 1) % 20 == 0:
        print(f"Обработано: {i+1}, пар: {len(paired_data)}/{RESUME_TARGET_PAIRS}")

    if len(paired_data) > start_count and len(paired_data) % 10 == 0:
        with open(PAIRED_PATH, "w", encoding="utf-8") as f:
            json.dump(paired_data, f, ensure_ascii=False, indent=2)

    time.sleep(SLEEP_SECONDS)

with open(PAIRED_PATH, "w", encoding="utf-8") as f:
    json.dump(paired_data, f, ensure_ascii=False, indent=2)

print("Было пар:", start_count)
print("Стало пар:", len(paired_data))
print("Новых пар:", len(paired_data) - start_count)


Уже есть пар: 57
Кандидатов до sample: 643
Кандидатов в этом запуске: 500
Цель: 150
Обработано: 20, пар: 68/150
Обработано: 40, пар: 73/150
Обработано: 60, пар: 82/150
Обработано: 80, пар: 90/150
Обработано: 100, пар: 98/150
Обработано: 120, пар: 104/150
Обработано: 140, пар: 112/150
Обработано: 160, пар: 122/150
Обработано: 180, пар: 133/150
Обработано: 200, пар: 143/150
Было пар: 57
Стало пар: 150
Новых пар: 93


## Контроль коротких позитивных real-отзывов

Сюда попадали тексты вроде “НЕ рекомендую”, потому что проверялось просто наличие слова рекомендую. Здесь отрицание исключается. Эти отзывы используются только для аудита, не добавляются отдельным классом


In [12]:
def is_short_positive_real(text):
    t = f' {normalize_text(text).lower()} '
    has_positive = any(w in t for w in POSITIVE_MARKERS)
    has_negative = any(w in t for w in NEGATIVE_MARKERS) or ' не рекомендую' in t or 'не советую' in t
    return len(text) < 90 and has_positive and not has_negative

short_positive_real = [t for t in real_wb_df['text'].tolist() if is_short_positive_real(t)]
print('Коротких позитивных real-отзывов для аудита:', len(short_positive_real))
for t in short_positive_real[:5]:
    print(f'[{len(t)}] {t}')


Коротких позитивных real-отзывов для аудита: 8
[81] Все пришло хорошо упакованым. Ну вкус чувствуется химия очень сильно. На любителя
[86] Данный товар прослужил ровно 3 часа с отдыхом каждые 15 мин работы. Товаром не доволен
[62] Молоко не вкусное, хорошо что на пробу взяли только одну пачку
[82] Ничего не выросло, все делали по инструкции. Товар пришел быстро и хорошо упакован
[65] Отличный шоколад, остались очень довольны. Будем заказывать ещё .


## Сборка согласованного датасета

In [16]:
if 'paired_data' not in globals():
    with open(PAIRED_PATH, encoding='utf-8') as f:
        paired_data = json.load(f)

pairs_df = pd.DataFrame(paired_data)
required_cols = {'pair_id', 'real', 'fake'}
missing = required_cols - set(pairs_df.columns)
if missing:
    raise ValueError(f'В paired_reviews.json не хватает колонок: {missing}')

def relaxed_quality_pair(real_text, fake_text):
    real_text = normalize_text(real_text)
    fake_text = normalize_text(fake_text)

    if fake_text == real_text:
        return False
    if len(fake_text) < 30:
        return False

    audit = pair_audit(real_text, fake_text)

    length_ok = 0.50 <= audit["len_ratio"] <= 2.00
    template_ok = audit["fake_template_count"] <= 3

    return length_ok and template_ok


audit_rows = []
for _, row in pairs_df.iterrows():
    audit = pair_audit(row["real"], row["fake"])
    audit["quality_ok"] = relaxed_quality_pair(row["real"], row["fake"])
    audit_rows.append(audit)

audit_df = pd.DataFrame(audit_rows)
pairs_df = pd.concat([
    pairs_df.drop(columns=[c for c in audit_df.columns if c in pairs_df.columns], errors='ignore'),
    audit_df
], axis=1)

filtered_pairs = pairs_df[pairs_df['quality_ok']].copy().reset_index(drop=True)
print('Пар до фильтрации:', len(pairs_df))
print('Пар после фильтрации:', len(filtered_pairs))
if len(filtered_pairs) < 200:
    print('ВНИМАНИЕ: качественных пар мало. Лучше догенерировать данные новым prompt, а не ослаблять фильтры.')

meta_cols = ['pair_id', 'rating', 'product_name', 'category', 'generator_model', 'prompt_version',
             'real_len', 'fake_len', 'len_ratio', 'real_bucket', 'fake_bucket']
meta_cols = [c for c in meta_cols if c in filtered_pairs.columns]

real_rows = filtered_pairs[meta_cols + ['real']].rename(columns={'real': 'text'})
real_rows['is_fake'] = 0
real_rows['source'] = 'wildberries_real'

fake_rows = filtered_pairs[meta_cols + ['fake']].rename(columns={'fake': 'text'})
fake_rows['is_fake'] = 1
fake_rows['source'] = 'synthetic_llm_tone_matched'

dataset_df = pd.concat([real_rows, fake_rows], ignore_index=True)
dataset_df['text'] = dataset_df['text'].map(normalize_text)
dataset_df = dataset_df.drop_duplicates(subset=['text']).sample(frac=1, random_state=SEED).reset_index(drop=True)

pair_counts = dataset_df.groupby('pair_id')['is_fake'].nunique()
complete_pair_ids = pair_counts[pair_counts == 2].index
dataset_df = dataset_df[dataset_df['pair_id'].isin(complete_pair_ids)].reset_index(drop=True)

print('Итоговый датасет:', len(dataset_df), 'строк')
print(dataset_df['is_fake'].value_counts().rename({0: 'real', 1: 'fake'}))
print(dataset_df.assign(text_len=dataset_df['text'].str.len()).groupby('is_fake')['text_len'].describe().round(1))
display(dataset_df.sample(min(6, len(dataset_df)), random_state=SEED)[['pair_id', 'text', 'is_fake', 'source']])


Пар до фильтрации: 150
Пар после фильтрации: 150
ВНИМАНИЕ: качественных пар мало. Лучше догенерировать данные новым prompt, а не ослаблять фильтры.
Итоговый датасет: 300 строк
is_fake
fake    150
real    150
Name: count, dtype: int64
         count   mean    std   min    25%    50%    75%    max
is_fake                                                       
0        150.0  227.1   97.1  47.0  144.2  213.0  305.2  440.0
1        150.0  317.4  103.7  74.0  241.5  325.5  389.8  611.0


,pair_id,text,is_fake,source
203,wb_pair_00054,"Заказывала пасту, честно переживала после того...",1,synthetic_llm_tone_matched
266,wb_pair_00144,"Пришло быстро, упаковка нормальная. Банка цела...",1,synthetic_llm_tone_matched
152,wb_pair_00073,"Взяла эти мешки для своего Филипса FC8132, в п...",1,synthetic_llm_tone_matched
9,wb_pair_00025,"Пришло всё правильно, но сначала думала что не...",1,synthetic_llm_tone_matched
233,wb_pair_00108,"лупа пришла нормальная, стекло целое, увеличив...",1,synthetic_llm_tone_matched
226,wb_pair_00110,Паста пришла расколотая вдребезги!!!! Была защ...,0,wildberries_real


In [17]:
dataset_df.to_csv('paired_wb_fake.csv', index=False, encoding='utf-8')

fake_export = (
    dataset_df[dataset_df['is_fake'] == 1]
    .sort_values('pair_id')
    .to_dict(orient='records')
)
with open('fake_wb_final.json', 'w', encoding='utf-8') as f:
    json.dump(fake_export, f, ensure_ascii=False, indent=2)

filtered_pairs.to_json('paired_reviews.json', orient='records', force_ascii=False, indent=2)